1: Load Data and Explore Missingness

Step 1.1 Import Libraries

In [1]:
import pandas as pd
import numpy as np

Step 1.2 Load the Dataset

In [2]:
df = pd.read_csv("credit_applicants.csv")

print(df.shape)
df.head()

(400, 10)


,applicant_id,age,monthly_income_inr,existing_loans_count,credit_utilization_ratio,upi_monthly_inflow_inr,bounced_payments_count,credit_bureau_score,employment_type,default
0,APP1000,59,41646,4,0.80,17398,2,NaN,salaried,1
1,APP1001,49,119185,0,0.08,58503,3,380.0,salaried,0
2,APP1002,35,38049,0,0.59,8638,1,788.0,salaried,1
3,APP1003,28,113116,0,0.26,8570,2,387.0,salaried,0
4,APP1004,41,112379,2,0.16,70785,3,493.0,salaried,0


Step 1.3 Calculate Default Rate

In [3]:
default_rate = df["default"].mean() * 100

print(f"Default Rate: {default_rate:.2f}%")

Default Rate: 20.25%


Step 1.4 Calculate Missing Bureau Score Percentage

In [4]:
missing_pct = (
    df["credit_bureau_score"]
    .isna()
    .mean()
    * 100
)

print(
    f"Missing Bureau Score Percentage: {missing_pct:.2f}%"
)

Missing Bureau Score Percentage: 20.00%


Step 1.5 Create Thin-File Indicator

In [5]:
df["is_thin_file"] = (
    df["credit_bureau_score"]
    .isna()
    .astype(int)
)

df[
    ["credit_bureau_score",
     "is_thin_file"]
].head()

,credit_bureau_score,is_thin_file
0,NaN,1
1,380.0,0
2,788.0,0
3,387.0,0
4,493.0,0


Task 2: Train-Test Split

Step 2.1 Separate Features and Target

In [6]:
X = df.drop("default", axis=1)

y = df["default"]

Step 2.2 Perform Stratified Split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )
)

Verify Stratification

In [8]:
print("Overall Default Rate:")
print(y.mean())

print("\nTraining Default Rate:")
print(y_train.mean())

print("\nTesting Default Rate:")
print(y_test.mean())

Overall Default Rate:
0.2025

Training Default Rate:
0.20333333333333334

Testing Default Rate:
0.2


Task 2.3 Median Imputation (Training Data Only)

Calculate Median from Training Split

In [9]:
bureau_median = (
    X_train["credit_bureau_score"]
    .median()
)

print(
    f"Training Median Bureau Score: {bureau_median}"
)


Training Median Bureau Score: 612.0


Fill Missing Values

In [10]:
# Training:
X_train["credit_bureau_score"] = (
    X_train["credit_bureau_score"]
    .fillna(bureau_median)
)

# Testing:
X_test["credit_bureau_score"] = (
    X_test["credit_bureau_score"]
    .fillna(bureau_median)
)

Task 2.4 Encode Employment Type

One-Hot Encoding

In [11]:
X_train = pd.get_dummies(
    X_train,
    columns=["employment_type"],
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=["employment_type"],
    drop_first=True
)

Align Columns

In [12]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

Task 2.5 Scale Numeric Features

In [13]:
numeric_cols = [
    "age",
    "monthly_income_inr",
    "existing_loans_count",
    "credit_utilization_ratio",
    "upi_monthly_inflow_inr",
    "bounced_payments_count",
    "credit_bureau_score"
]

Fit StandardScaler on Training Data Only

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

Final Verification

In [15]:
print(X_train.shape)
print(X_test.shape)

X_train.head()

(300, 11)
(100, 11)


,applicant_id,age,monthly_income_inr,existing_loans_count,credit_utilization_ratio,upi_monthly_inflow_inr,bounced_payments_count,credit_bureau_score,is_thin_file,employment_type_salaried,employment_type_self_employed
84,APP1084,-1.145762,0.952812,0.053277,0.149735,0.995851,-0.227884,-0.148387,0,False,True
10,APP1010,0.270945,-1.385434,-0.641639,-0.587877,-1.151228,-0.227884,0.048994,1,True,False
384,APP1384,0.713666,-1.586508,-0.641639,-1.436132,0.076032,-1.127427,0.127947,0,False,True
359,APP1359,-0.171776,0.287041,0.053277,1.588080,-0.656830,-0.227884,1.338550,0,False,False
11,APP1011,1.333476,-0.576811,1.443108,0.186616,1.211575,-0.227884,0.476653,0,True,False
